# 데이터셋 전처리

`260526_create_dataset`에서 생성한 xlsx를 임상적으로 유도 가능한 값으로 보완합니다.


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from core.preprocessing_pipeline import PreprocessingConfig, PreprocessingPipeline

INPUT_DIR = Path("../outputs/260526_create_dataset")
OUTPUT_DIR = Path("../outputs/260526_preprocessing")
MISSING_RATE_THRESHOLD = 0.20  # 이상 결측 비율 (0.20 = 20%)
MIN_NUMERIC_RATIO = 0.90  # 비결측 중 이 비율 이상 수치 해석 가능 시 텍스트 공백

CONFIG = PreprocessingConfig(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    missing_rate_threshold=MISSING_RATE_THRESHOLD,
    min_numeric_ratio=MIN_NUMERIC_RATIO,
)


## 1. 전처리

| 대상 | 규칙 |
|------|------|
| (수치형 인자) | 비결측 값의 **90% 이상**이 수치로 해석되면, 나머지 **텍스트는 공백** 후 **numeric** 변환 (요검사·메타 제외) |
| `혈압(수축기)`, `혈압(이완기)` | `이완기` ≥ `수축기`이면 두 값 교환 (컬럼 반대 기입) |
| `WBC` | K/µL. 값 ≥ 1000이면 /µL 오입력으로 보고 **÷1000**, 소수 둘째 자리 |
| `RBC` | M/µL. 값 ≥ 100이면 /µL 오입력으로 보고 **÷100**, 소수 둘째 자리 |
| `MCH`, `MCV` | `RBC`·`Hgb`·`Hct`·`MCV`·`MCH`·`MCHC` **6개 모두 유효**할 때, 아래 **뒤바뀜** 패턴이면 두 값 **교환**  |

**MCH·MCV 뒤바뀜 감지** (`RBC` ≠ 0, 허용 오차 ±0.5)

| 구분 | 조건 |
|------|------|
| 정상 | `\|MCH - Hgb/RBC×10\| ≤ 0.5` **그리고** `\|MCV - Hct/RBC×10\| ≤ 0.5` |
| 뒤바뀜 | 정상이 아니면서 `\|MCV - Hgb/RBC×10\| ≤ 0.5` **그리고** `\|MCH - Hct/RBC×10\| ≤ 0.5` → `MCH`↔`MCV` 교환 |


In [2]:
pipeline = PreprocessingPipeline(CONFIG)
pipeline.run_early_corrections()



[pre_diabetes] rows=3,113, cols=388
— 수치형 텍스트 공백 처리 (비결측 중 ≥90% 수치 해석 가능)


,field,numeric_ratio,blanked_count,dtype_before,dtype_after,excluded_values
0,AFP,0.9612,93,object,float64,"음성 (68), Negative (14), 음성(negative) (6), < 2.00 (2), < 1.0 (1), (1), Negative 1.03 (1)"
1,CA19-9,0.9887,20,object,float64,"< 2.0 (6), <2.00 (4), (3), < 2.06 (2), < 0.01 (1), <2.06 (1), < 2.00 (1), 0.80 이하 (1), <2.7 (1)"
2,CEA,0.9873,24,object,float64,"음성 (10), < 2.4 (3), 음성(negative) (3), < 0.01 (3), <1.73 (1), (1), <2.90 (1), 양성 (1), < 1.73 (1)"
3,N.RBC,0.9873,5,object,float64,"0-3 (2), 0-5 (2), 0-1 (1)"
4,기타 골밀도 Tscore,0.9375,9,object,float64,"정상 (3), 골밀도 검사결과 골밀도 수치가 정상범위보다 약간 낮게 측정된 골감소 ( -1.2 ) 소견입니다.\n\n골다공증 예방을 위해 칼슘이 풍부한 멸치, 유제품 등의 음식을 섭취하시고 유산소 운동을 \n\n규칙적으로 하십시오. (1), 정상 소견입니다.Region Z-scoreL1-L4 -0.9 (1), 정상 소견입니다.Region Z-scoreL1-L4 0.2 (1), 골밀도 측정 검사 결과 골감소증 소견을 보입니다. 골다공증 예방을 위해 칼슘제 복용 또는 칼슘이 풍부한 멸치, 뼈째먹는 생선, 유제품 등의 음식을 섭취하시고 규칙적인 운동을 하십시요. 1년 후 추적검사를 권합니다.(T-score(L1-4):-1.3) (1), 정상 소견입니다.\nRegion Z-score\nL1-L4 2.0 (1), 정상 소견입니다.\nRegion Z-score\nL1-L4 -1.7 (1)"
5,나이,0.9906,29,object,float64,"M (16), F (13)"
6,비만도,0.9385,121,object,float64,"정상체중 (55), 비만1단계 (26), 정상 (10), 과체중 (10), 비만 (7), 표준 (3), 비만(복부비만) (2), (비만1단계) (1), 03-비만 (1), 비만2단계 (1), 저체중 (1), 1단계 비만 (1), 경도비만 (1), 23.21(과체중) (1), 22.82(정상체중) (1)"
7,신장,0.9997,1,object,float64,174.1 Cm (1)
8,체중,0.9997,1,object,float64,59.8 Kg (1)


— 혈압 반대 기입 교환 (수축기/이완기)


,field,swapped_count,swap_detail
0,혈압(수축기)·혈압(이완기),5,"61/105 → 105/61 (1), 75/115 → 115/75 (1), 68/110 → 110/68 (1), 86/115 → 115/86 (1), 5/65 → 65/5 (1)"


— WBC 단위 보정 (≥1000 → ÷1000, K/µL)


...

,field,fixed_count,fix_detail
0,WBC,13,"6320 → 6.32 (2), 5160 → 5.16 (1), 5120 → 5.12 (1), 4200 → 4.20 (1), 4630 → 4.63 (1), 8380 → 8.38 (1), 10860 → 10.86 (1), 9010 → 9.01 (1), 5700 → 5.70 (1), 7200 → 7.20 (1), 2470 → 2.47 (1), 6990 → 6.99 (1)"


— RBC 단위 보정 (≥100 → ÷100, M/µL)


,field,fixed_count,fix_detail
0,RBC,13,"485 → 4.85 (2), 505 → 5.05 (1), 502 → 5.02 (1), 443 → 4.43 (1), 459 → 4.59 (1), 521 → 5.21 (1), 506 → 5.06 (1), 476 → 4.76 (1), 574 → 5.74 (1), 483 → 4.83 (1), 458 → 4.58 (1), 532 → 5.32 (1)"


— MCH·MCV 뒤바뀜 교환


,field,swapped_count,swap_detail
0,MCH·MCV,5,"MCH 91.1→30.9 / MCV 30.9→91.1 (1), MCH 90.2→30.2 / MCV 30.2→90.2 (1), MCH 93→31.4 / MCV 31.4→93 (1), MCH 91.3→32.1 / MCV 32.1→91.3 (1), MCH 91.4→31 / MCV 31→91.4 (1)"


| 대상 | 변환 |
|------|------|
| `Bilirubin`, `Blood`, `Glucose`, `Keton`, `Leukocyte`, `Nitrite`, `Protein`, `Urobilinogen`, `HBs-Ab`, `HBs-Ag` | `음성` / `양성` (분류 불가 → 결측) |


In [3]:
pipeline.run_lab_encoding()



— Bilirubin
  양성: 양성(1+) (14), 1 Positive (7), +- (6), 양성(+) (5), 양성 (2), + (1), 1+ (1), 약양성 (1), 양성(++) (1), 약양성(+/-) (1), 양성(+2) (1)
  음성: 음성 (2419), Negative (249), - (155), 음 성 (39), 음성(-) (20), 음성(negative) (12), neg (5), 01-음성 (1), 음성  (1)
  NaN: NaN (171), 1.00 (1)

— Blood
  양성: 약양성(+/-) (62), 양성(+) (51), 양성(1+) (49), 양성(2+) (33), 약양성 (30), 양성(3+) (30), 양성(+++) (21), 양성(+1) (18), + (15), 양성 (13), Trace (12), 양성(++) (11), 약양성(+-) (10), 2+ (6), 양성(+2) (5), 1 Positive (4), ++ (3), 양성(4+) (3), 4 Positive (3), 약양성(±) (3), 2 Positive (3), 양성(+3) (2), 양성(++++) (2), +- (2), 1+ (2), 3 양성 (2), +/- (2), + 3 (1), 3+ (1), 3 Positive (1), 2Positive (1), 2 양성 (1), 1 양성 (1), 1Positive (1), 1Positive  (1), +3 (1), 약양성 * (1), Weakly Positive(+/-) (1), Trace(+-) (1), 양성(positive)(++++) (1)
  음성: 음성 (2033), Negative (218), - (141), 음 성 (56), 음성(-) (17), 음성(negative) (11), neg (2), 01-음성 (1), 움성 (1)
  NaN: NaN (220), 3 (1), ± (5) (1), 4 (1)

— Glucose
  양성: 3+ (1), +- (1), ++++ (1), 3 Positive (1)

### 임상 불가능 수치 공백 처리 (생리학적·최대 보수)

정성 검사 인코딩 **직후**, 임상·결측 보완 **이전**·**§2 고결측 제외 이전**에 **수치형 인자**에 대해 **생리학적으로 불가능하거나 명백한 입력 오류**만 결측(공백)으로 둡니다.

- **최대 보수 원칙**: 생리학적으로 나올 수 있는 범위를 넓게 두고, 경계성·고위험 비정상 수치는 **유지**
- **제거 대상**: 센티널(`999` 계열), 혈압 논리 오류(전처리 교환 후 잔여), 생리학적 불가능 극단값 (`WBC` ≥1000·`RBC` ≥100 /µL 오입력은 §1에서 보정)

### 공통 (항상 적용)

| 구분 | 기준 |
|------|------|
| 센티널 | `999`, `999.9`, `9999`, `99999` |
| 논리 오류 | `혈압(이완기)` ≥ `혈압(수축기)` → 두 혈압 모두 공백 |

### 인자별 허용 범위 (`ClinicalRangeBlanker.DEFAULT_RULES`, 최대 보수)

| 구분 | 인자 | 최소 | 최대 | 비고 |
|------|------|------|------|------|
| 인구·체형 | `나이` | 0 | 150 | 세 |
|  | `신장` | 50 | 250 | cm |
|  | `체중` | 10 | 500 | kg |
|  | `BMI` | 5 | 120 | kg/m² |
|  | `비만도` | 0 | 500 | % |
|  | `허리둘레` | 10 | 500 | cm |
| 혈압 | `혈압(수축기)` | 50 | 250 | mmHg |
|  | `혈압(이완기)` | 30 | 150 | mmHg |
| 당뇨 | `공복혈당` | 1 | 1000 | mg/dL |
| 지질 | `T.Cholesterol` | 30 | 1000 | — |
|  | `HDL` | 1 | 300 | — |
|  | `LDL` | 1 | 600 | — |
|  | `Triglyceride` | 1 | 10000 | — |
| 간 | `GOT(AST)`, `GPT(ALT)` | 0 | 10000 | U/L |
|  | `r-GTP` | 0 | 5000 | — |
|  | `ALP` | 0 | 2000 | — |
|  | `T.Bilirubin` | 0 | 50 | — |
|  | `D.Bilirubin` | 0 | 30 | — |
| 신장 | `Creatinine` | 0.1 | 30 | — |
|  | `e-GFR` | 0 | 500 | mL/min/1.73m² |
|  | `BUN` | 0.5 | 200 | — |
|  | `Uric acid` | 0.1 | 30 | — |
| 혈액 | `WBC` | 0.1 | 200 | K/uL |
|  | `RBC` | 0.5 | 15 | M/uL |
|  | `Hgb` | 1 | 30 | — |
|  | `Hct` | 5 | 90 | — |
|  | `Platelet` | 1 | 5000 | 10³/μL |
|  | `MCV` | 30 | 150 | — |
|  | `MCH` | 10 | 100 | — |
|  | `MCHC` | 20 | 50 | — |
|  | `RDW` | 3 | 60 | — |
|  | `MPV` | 3 | 25 | — |
|  | `PDW` | 5 | 80 | — |
| 혈액 | `Lymphocyte`, `Monocyte`, `Eosinophil`, `Basophil` | 0 | 100 | % |
|  | `B/C ratio` | 0 | 100 | — |
| 소변 | `PH` | 3 | 14 | — |
|  | `SG` | 1.0 | 1.06 | — |
| 기타 | `TSH` | 0 | 500 | — |
|  | `T.Protein` | 2 | 15 | — |
|  | `Albumin` | 1 | 8 | — |
|  | `Globulin` | 0.5 | 10 | — |
|  | `A/G ratio` | 0.1 | 25 | — |

`음성`/`양성` 인코딩 인자, `gender`, `label`, `interval_days` 등은 대상에서 제외합니다.


In [4]:
pipeline.run_range_blanking() 


[pre_diabetes] 임상 불가능 수치 공백 처리


,field,below_min,above_max,sentinel,logical,total_blanked,excluded_values
0,Hgb,2,1,0,0,3,"50.8 (1), 0.9 (1), 0.4 (1)"
1,허리둘레,0,3,3,0,3,999.9 (3)
2,Hct,1,1,0,0,2,"3.2 (1), 91.4 (1)"
3,체중,0,2,2,0,2,999.9 (2)
4,BMI,1,0,0,0,1,0 (1)
5,혈압(이완기),1,0,0,0,1,5 (1)
6,RBC,1,0,0,0,1,0.4 (1)
7,MCHC,1,0,0,0,1,15.6 (1)
8,PH,0,1,0,0,1,55 (1)
9,Globulin,1,0,0,0,1,0.4 (1)



[diabetes] 임상 불가능 수치 공백 처리


,field,below_min,above_max,sentinel,logical,total_blanked,excluded_values
0,Hgb,0,2,0,0,2,"41.7 (1), 46.1 (1)"
1,A/G ratio,0,2,0,0,2,"30 (1), 50 (1)"
2,MCV,2,0,0,0,2,"8.6 (1), 9.2 (1)"
3,LDL,1,0,0,0,1,0 (1)
4,BMI,0,1,0,0,1,123 (1)
5,PH,0,1,0,0,1,55 (1)
6,Albumin,0,1,0,0,1,8.3 (1)


### 임상·결측 보완

공백 처리 **이후**에 적용합니다. 입력 수치에 센티널(`999`, `999.9`, `9999`, `99999`)이 있으면 해당 행의 대체(BMI·지질·e-GFR 등)는 수행하지 않습니다.

결측치 처리

| 대상 컬럼 | 규칙 |
|-----------|------|
| `나이` | `나이` = (`checkup_date` - `birthday`).days / 365.25 |
| `gender` | `gender` 1·`성별` M/남/남자 → `남자`, `gender` 2·`성별` F/여/여자 → `여자`. 통합 후 `성별` 제거 |
| `BMI` | `BMI` = `체중` / (`신장` / 100)² |
| `T.Cholesterol` | `T.Cholesterol` = `LDL` + `HDL` + int(`Triglyceride`/5), `Triglyceride` < 400 |
| `HDL` | `HDL` = `T.Cholesterol` - `LDL` - int(`Triglyceride`/5), `Triglyceride` < 400 |
| `Triglyceride` | `Triglyceride` = 5×(`T.Cholesterol`-`LDL`-`HDL`), 결과 < 400 |
| `LDL` | `LDL` = `T.Cholesterol` - `HDL` - int(`Triglyceride`/5), `Triglyceride` < 400 |
| `Globulin` | `Globulin` = `T.Protein` - `Albumin` |
| `Albumin` | `Albumin` = `T.Protein` - `Globulin` |
| `T.Protein` | `T.Protein` = `Albumin` + `Globulin` |
| `A/G ratio` | `A/G ratio` = `Albumin` / `Globulin` |
| `UIBC` | `UIBC` = `TIBC` - `Fe` |
| `철포화율` | `철포화율` = `Fe` / `TIBC` × 100 |
| `비만도` | `비만도` = (`체중` / 표준체중) × 100, 표준체중 = (`신장` - 100) × (0.9 if `gender`=`남자` else 0.85) |
| `e-GFR` | CKD-EPI(2009) 공식으로 계산 (아래 참고) |
| `MCH` | `MCH` = `Hgb` / `RBC` × 10 (`RBC` ≠ 0) |
| `MCHC` | `MCHC` = `Hgb` / `Hct` × 100 (`Hct` ≠ 0) |
| `MCV` | `MCV` = `Hct` / `RBC` × 10 (`RBC` ≠ 0) |
| `Hct` | `Hct` = `RBC` × `MCV` / 10 |


### `e-GFR` — CKD-EPI(2009)


**대상·입력 컬럼**

| 구분 | 컬럼 |
|------|------|
| 채우는 컬럼 | `e-GFR` (이미 값이 있으면 덮어쓰지 않음) |
| 입력 | `Creatinine`, `나이`, `gender` (`남자` / `여자`) |


**기호 (`Creatinine` 단위: mg/dL)**

| 기호 | 의미 |
|------|------|
| $Cr$ | `Creatinine` |
| $Age$ | `나이` |
| $\kappa,\ \alpha,\ s$ | `gender`에 따른 상수 (아래 표) |

**성별별 상수**

| | `gender` = `여자` | `gender` = `남자` |
|--|--|--|
| $\kappa$ | 0.7 | 0.9 |
| $\alpha$ | −0.329 | −0.411 |
| $s$ (성별계수) | 1.018 | 1.0 |

**공식 (`core/preprocessor.py` → `_ckd_epi_2009`와 동일)**

$Cr \le \kappa$ 일 때:

$$eGFR = 141 \times \min(Cr/\kappa, 1)^{\alpha} \times \max(Cr/\kappa, 1)^{-0.329} \times 0.993^{Age} \times s$$

$Cr > \kappa$ 일 때:

$$eGFR = 141 \times \min(Cr/\kappa, 1)^{-1.209} \times \max(Cr/\kappa, 1)^{-1.209} \times 0.993^{Age} \times s$$


In [5]:
pipeline.run_imputations()


[pre_diabetes] 임상·결측 보완


,field,filled_count,rule
0,나이,70,나이 = (`checkup_date` - `birthday`).days / 365.25
1,BMI,404,BMI = 체중 / (신장/100)²
2,LDL,18,LDL = T.Cholesterol - HDL - int(Triglyceride/5...
3,T.Cholesterol,2,T.Cholesterol = LDL + HDL + int(Triglyceride/5...
4,HDL,20,HDL = T.Cholesterol - LDL - int(Triglyceride/5...
5,Triglyceride,11,"Triglyceride = 5×(T.Cholesterol-LDL-HDL), Trig..."
6,Globulin,153,Globulin = T.Protein - Albumin
7,Albumin,2,Albumin = T.Protein - Globulin
8,T.Protein,6,T.Protein = Albumin + Globulin
9,A/G ratio,189,A/G ratio = Albumin / Globulin


— 결측 비율 (공백 처리 후 → 임상 보완 후)


,field,rows,missing_before,missing_pct_before,missing_after,missing_pct_after,filled,pct_point_change
0,나이,3113,70,2.25%,0,0.00%,70,-2.25%p
1,BMI,3113,415,13.33%,11,0.35%,404,-12.98%p
2,T.Cholesterol,3113,38,1.22%,36,1.16%,2,-0.06%p
3,HDL,3113,109,3.50%,89,2.86%,20,-0.64%p
4,Triglyceride,3113,49,1.57%,38,1.22%,11,-0.35%p
5,LDL,3113,111,3.57%,93,2.99%,18,-0.58%p
6,Globulin,3113,282,9.06%,129,4.14%,153,-4.92%p
7,Albumin,3113,119,3.82%,117,3.76%,2,-0.06%p
8,T.Protein,3113,135,4.34%,129,4.14%,6,-0.20%p
9,A/G ratio,3113,317,10.18%,128,4.11%,189,-6.07%p



[diabetes] 임상·결측 보완


,field,filled_count,rule
0,나이,27,나이 = (`checkup_date` - `birthday`).days / 365.25
1,BMI,245,BMI = 체중 / (신장/100)²
2,LDL,14,LDL = T.Cholesterol - HDL - int(Triglyceride/5...
3,HDL,4,HDL = T.Cholesterol - LDL - int(Triglyceride/5...
4,Triglyceride,3,"Triglyceride = 5×(T.Cholesterol-LDL-HDL), Trig..."
5,Globulin,90,Globulin = T.Protein - Albumin
6,Albumin,1,Albumin = T.Protein - Globulin
7,T.Protein,8,T.Protein = Albumin + Globulin
8,A/G ratio,108,A/G ratio = Albumin / Globulin
9,UIBC,66,UIBC = TIBC - Fe


— 결측 비율 (공백 처리 후 → 임상 보완 후)


,field,rows,missing_before,missing_pct_before,missing_after,missing_pct_after,filled,pct_point_change
0,나이,1458,27,1.85%,0,0.00%,27,-1.85%p
1,BMI,1458,249,17.08%,4,0.27%,245,-16.81%p
2,T.Cholesterol,1458,14,0.96%,14,0.96%,0,+0.00%p
3,HDL,1458,26,1.78%,22,1.51%,4,-0.27%p
4,Triglyceride,1458,17,1.17%,14,0.96%,3,-0.21%p
5,LDL,1458,40,2.74%,26,1.78%,14,-0.96%p
6,Globulin,1458,147,10.08%,57,3.91%,90,-6.17%p
7,Albumin,1458,55,3.77%,54,3.70%,1,-0.07%p
8,T.Protein,1458,66,4.53%,58,3.98%,8,-0.55%p
9,A/G ratio,1458,164,11.25%,56,3.84%,108,-7.41%p


In [6]:
pipeline.run_range_blanking() 


[pre_diabetes] 임상 불가능 수치 공백 처리
  공백 처리된 인자 없음

[diabetes] 임상 불가능 수치 공백 처리
  공백 처리된 인자 없음


## 2. 고결측 컬럼 제외

아래 코드 셀의 **설정**에서 결측 임계값·유지 컬럼을 조정합니다. 메타 컬럼(`user_key`, `label`, `selected_transition`, `full_transition` 등)과 `HBs-Ab`, `HBs-Ag`는 결측률과 관계없이 유지됩니다.


In [7]:
# ===== 고결측 컬럼 제외 설정 =====
EXTRA_PRESERVE_COLUMNS: list[str] = []  # 고결측 제외 대상에서 제외(항상 유지)
EXTRA_ALWAYS_DROP_COLUMNS: list[str] = [
    "흉부 X-RAY(정면)",
    "심전도",
    "청력우",
    "청력좌",
    "birthday",
]  # 결측률과 무관하게 제외

pipeline.configure_column_filter(
    extra_preserve_columns=frozenset(EXTRA_PRESERVE_COLUMNS),
    extra_always_drop_columns=frozenset(EXTRA_ALWAYS_DROP_COLUMNS),
)
pipeline.run_column_filter()


항상 제외 컬럼: ['birthday', '심전도', '청력우', '청력좌', '흉부 X-RAY(정면)']

[pre_diabetes] 결측 ≥ 20% 인자 제외
  컬럼 수: 387 → 66 (제거 321개)


,field,missing_count,missing_pct
0,STD(성매개질환),3112,99.97
1,% free PSA,3112,99.97
2,Free Fatty Acid,3112,99.97
3,IGF-1,3112,99.97
4,Measles-IgG,3112,99.97
...,...,...,...
316,백혈구(소변현미경),714,22.94
317,청력우,554,17.80
318,청력좌,552,17.73
319,심전도,283,9.09



[diabetes] 결측 ≥ 20% 인자 제외
  컬럼 수: 366 → 64 (제거 302개)


,field,missing_count,missing_pct
0,% free PSA,1457,99.93
1,ACTH(basal),1457,99.93
2,Anti Thyroglobulin Ab,1457,99.93
3,FEF75-85%(L/S),1457,99.93
4,LH,1457,99.93
...,...,...,...
297,청력좌,312,21.40
298,HBs-Ag,304,20.85
299,적혈구(소변현미경),294,20.16
300,심전도,136,9.33


## 3. 저장


In [8]:
pipeline.run_export()



[pre_diabetes] 저장 전 DataFrame
  행=3,113, 열=66
  numeric (48개): label, interval_days, A/G ratio, ALP, Albumin, B/C ratio, BMI, BUN, Basophil, Creatinine, D.Bilirubin, Eosinophil, GOT(AST), GPT(ALT), Globulin, HDL, Hct, Hgb, LDL, Lymphocyte, MCH, MCHC, MCV, MPV, Monocyte, PDW, PH, Platelet, RBC, RDW, SG, T.Bilirubin, T.Cholesterol, T.Protein, TSH, Triglyceride, Uric acid, WBC, e-GFR, r-GTP, 공복혈당, 나이, 비만도, 신장, 체중, 허리둘레, 혈압(수축기), 혈압(이완기)
  object (18개): user_key, current_checkup_date, future_checkup_date, selected_transition, full_transition, checkup_date, gender, birthday, Bilirubin, Blood, Glucose, HBs-Ab, HBs-Ag, Keton, Leukocyte, Nitrite, Protein, Urobilinogen
Saved: ../outputs/260526_preprocessing/pre_diabetes_dataset.xlsx  (3,113명, 66컬럼)

[diabetes] 저장 전 DataFrame
  행=1,458, 열=64
  numeric (48개): label, interval_days, A/G ratio, ALP, Albumin, B/C ratio, BMI, BUN, Basophil, Creatinine, D.Bilirubin, Eosinophil, GOT(AST), GPT(ALT), Globulin, HDL, Hct, Hgb, LDL, Lymphocyte, MCH, MCHC, M